# DART 사업보고서 정리: 사업의 내용

사업보고서 "II. 사업의 내용"에서 국내 기업의 **제품·원재료·매입처·판매처·경쟁사** 정보를 뽑기 위한 준비를 한다. 원칙과 항목은 `graphDB.md` 5장(5.2)을 따른다.

이 노트북이 하는 일은 두 가지다.

1. **LLM 입력 파일 만들기**: 사업의 내용에서 필요한 부분만 잘라 회사마다 글 파일 하나로 저장한다. 세부 제품, 영어 검색어, 경쟁사·계약 상대 같은 **해석이 필요한 항목은 다음 노트북(`dart_business_extractor.ipynb`)에서 LLM으로 뽑는다.**
2. **표를 규칙으로 읽기**: 사업부문별 매출 비중, 원재료별 매입 비중과 매입처, 사업장을 표에서 바로 읽는다. 이 값은 LLM에게 다시 시키지 않고, 그래프의 숫자 기준과 LLM 결과 점검에 쓴다.

- 이 노트북은 **서버에 요청하지 않는다.** `dart_collector`가 받아 둔 사업보고서 원문(`source/dart/raw/document/`)만 읽으므로 몇 번을 다시 실행해도 된다.
- 어느 파일을 읽을지는 `source/dart/selected_reports.csv`의 "사업보고서 원문" 줄을 따른다(기업마다 최신 사업보고서).
- 표는 회사마다 서식이 달라 규칙으로 다 읽히지 않는다. 읽히지 않은 표도 LLM 입력 파일에는 그대로 들어간다.

**입력**

| 파일 | 내용 |
|---|---|
| `source/dart/selected_reports.csv` | 기업마다 고른 사업보고서와 원문 zip 위치 |
| `source/dart/raw/document/*.zip` | 사업보고서 원문 |
| `source/dart/parsed/companies.csv` | 기업 이름, WI26 업종 (`dart_parser`) |

**출력** (모두 `source/dart/business/` 아래)

| 파일 | 한 줄 / 내용 |
|---|---|
| `inputs/종목코드.txt` | 회사 하나의 LLM 입력 글 |
| `inputs_index.csv` | 회사 하나: 서식, 부분별 글자 수, 뺀 영역 수 |
| `tables/segments.csv` | 사업부문·제품 하나: 매출액, 매출 비중 |
| `tables/materials.csv` | 원재료 하나: 매입액, 매입 비중, 매입처 |
| `tables/sites.csv` | 사업장 하나: 구분, 이름, 주소, 나라 |

**걸리는 시간**: 약 2분.

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `dart_collector` → `dart_parser` → **`dart_business_parser`** → `dart_business_extractor` |
| 외부 요청·비용 | 없음(받아 둔 원문만 읽는다) |
| 다시 실행 | 언제든 된다 |
| 처리 순서 | 설정 → 공통 함수 → 절 나누기 → 글로 바꾸고 줄이기 → LLM 입력 파일 쓰기 → 표 규칙 추출(매출 비중·원재료·사업장) → 결과 확인 |

## 1. 설정

In [1]:
# ── 설정: 폴더, 입력 글을 줄이는 값 ──
import json
import re
import zipfile
import warnings
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning

warnings.filterwarnings('ignore', category=XMLParsedAsHTMLWarning)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)

# 이 노트북은 graph_db 폴더에서 실행한다.
ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'

RAW = ROOT / 'source' / 'dart' / 'raw'
SELECTED_PATH = ROOT / 'source' / 'dart' / 'selected_reports.csv'
COMPANIES_PATH = ROOT / 'source' / 'dart' / 'parsed' / 'companies.csv'
OUT = ROOT / 'source' / 'dart' / 'business'
INPUTS = OUT / 'inputs'
TABLES = OUT / 'tables'
INPUTS.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

# 입력 글을 줄이는 값
MAX_ROWS = 20              # 표는 앞에서 이 줄 수까지만 남긴다(2절은 아래 값)
MAX_ROWS_PRODUCTS = 60     # 2절(주요 제품) 표
SHARE_WINDOW_CHARS = 2500  # 1·7절에서 시장점유율·경쟁 단락을 잘라 올 때 절마다 최대 글자 수

selected = pd.read_csv(SELECTED_PATH, dtype=str).fillna('')
selected = selected[selected['item'] == '사업보고서 원문'].reset_index(drop=True)
companies = pd.read_csv(COMPANIES_PATH, dtype=str).fillna('').set_index('stock_code')
print('사업보고서 원문', len(selected), '개 / 대상 기업', len(companies), '개')

사업보고서 원문 1063 개 / 대상 기업 1067 개


## 2. 공통 함수

- `table_grid`: 표의 병합된 칸(rowspan·colspan)을 펼친다. `dart_parser`와 같은 규칙이다.
- `country_of`: 주소에서 나라를 찾는다. `dart_parser`의 나라 사전을 그대로 쓴다.
- `to_num`: `1,234`, `(1,234)`, `△1,234`, `12.5%` 같은 표 숫자를 숫자로 바꾼다. 각주 표시(`(*1)`)는 지운다.

In [2]:
# ── 공통 함수: 원문 읽기, 표 펼치기, 숫자, 나라 찾기 ──
TITLE_RE = re.compile(r'<TITLE[^>]*>(.*?)</TITLE>', re.S)


def clean(text):
    """줄바꿈과 겹친 공백을 한 칸으로"""
    return re.sub(r'\s+', ' ', str(text or '')).strip()


def read_main(path):
    """원문 zip에서 본문 XML(가장 큰 파일)을 읽는다."""
    with zipfile.ZipFile(path) as z:
        main = max(z.namelist(), key=lambda n: z.getinfo(n).file_size)
        return z.read(main).decode('utf-8', 'ignore')


def period_end(report_nm):
    """'사업보고서 (2025.12)' → '2025-12-31'"""
    m = re.search(r'\((\d{4})\.(\d{2})\)', report_nm)
    return str(pd.Period(f'{m.group(1)}-{m.group(2)}', 'M').end_time.date()) if m else ''


def table_grid(table):
    """표를 rowspan·colspan을 펼친 칸 목록으로 바꾼다. [[칸, ...], ...]"""
    out, pending = [], {}
    for tr in table.find_all('tr'):
        row, col = [], 0
        cells = tr.find_all(['th', 'td', 'te', 'tu'], recursive=False)

        def fill_pending():
            nonlocal col
            while col in pending:
                text, left = pending[col]
                row.append(text)
                if left <= 1:
                    del pending[col]
                else:
                    pending[col] = (text, left - 1)
                col += 1

        for c in cells:
            fill_pending()
            text = clean(c.get_text(' '))
            colspan = int(c['colspan']) if str(c.get('colspan', '')).isdigit() else 1
            rowspan = int(c['rowspan']) if str(c.get('rowspan', '')).isdigit() else 1
            for _ in range(colspan):
                row.append(text)
                if rowspan > 1:
                    pending[col] = (text, rowspan - 1)
                col += 1
        fill_pending()
        if any(row):
            out.append(row)
    return out


NUM_CELL = re.compile(r'^[\(\-△]?\s*[\d,]+(\.\d+)?\s*\)?\s*%?$|^-$|^－$')


def to_num(text):
    """표 숫자 → float. 숫자가 아니면 None"""
    s = clean(text).replace(',', '').replace('△', '-').replace('%', '')
    s = re.sub(r'\(\s*\*.*?\)|\*\d*', '', s).strip()
    if s in ('', '-', '－'):
        return None
    negative = s.startswith('(') and s.endswith(')')
    try:
        v = float(s.strip('()'))
    except ValueError:
        return None
    return -v if negative else v


# 나라 이름 사전: 정규식 → 나라. 앞에 있을수록 먼저 본다.
COUNTRIES = [
    (r'대한민국|한\s*국|국\s*내|korea|서울|부산|대구|인천|광주광역|대전|울산|세종|경기|강원|충청|충북|충남|전라|전북|전남|경상|경북|경남|제주', '대한민국'),
    (r'미\s*국|u\.?s\.?a\b|united states|\bu\.s\.|america\b|wilmington|silicon valley', '미국'),
    (r'중\s*국|china|\bprc\b|beijing|shanghai|shenzhen|guangzhou|jiangsu|tianjin|북경|상해|천진|소주|강소', '중국'),
    (r'홍\s*콩|hong\s*kong', '홍콩'), (r'대\s*만|taiwan', '대만'), (r'일\s*본|japan|tokyo|osaka', '일본'),
    (r'베트남|vietnam|viet nam|hanoi|ho chi minh', '베트남'), (r'인도네시아|indonesia|jakarta', '인도네시아'),
    (r'싱가포르|싱가폴|singapore', '싱가포르'), (r'말레이시아|malaysia', '말레이시아'), (r'태\s*국|thailand|bangkok', '태국'),
    (r'필리핀|philippines', '필리핀'), (r'인\s*도(?!네)|india\b|mumbai|new delhi|chennai|bangalore', '인도'),
    (r'캄보디아|cambodia', '캄보디아'), (r'미얀마|myanmar', '미얀마'), (r'몽골|mongolia', '몽골'),
    (r'방글라데시|bangladesh', '방글라데시'), (r'파키스탄|pakistan', '파키스탄'), (r'스리랑카|sri lanka', '스리랑카'),
    (r'우즈베키스탄|uzbekistan', '우즈베키스탄'), (r'카자흐스탄|kazakhstan', '카자흐스탄'),
    (r'호\s*주|australia|\b(NSW|VIC|QLD)\s+\d{4}\b|sydney|melbourne', '호주'), (r'뉴질랜드|new zealand', '뉴질랜드'),
    (r'캐나다|canada', '캐나다'), (r'멕시코|mexico', '멕시코'), (r'브라질|brazil|brasil|s[ãa]o paulo|manaus', '브라질'),
    (r'칠레|chile', '칠레'), (r'페루|peru\b', '페루'), (r'아르헨티나|argentina', '아르헨티나'), (r'콜롬비아|colombia', '콜롬비아'),
    (r'파나마|panama', '파나마'),
    (r'영\s*국|united kingdom|\bu\.?k\.?\b|england|london|scotland', '영국'), (r'독\s*일|germany|deutschland|berlin|m[üu]nchen|munich|frankfurt|hamburg|düsseldorf|stuttgart', '독일'),
    (r'프랑스|france|paris', '프랑스'), (r'네덜란드|netherlands|amsterdam', '네덜란드'), (r'벨기에|belgium', '벨기에'),
    (r'룩셈부르크|luxembourg', '룩셈부르크'), (r'스위스|switzerland', '스위스'), (r'오스트리아|austria', '오스트리아'),
    (r'이탈리아|italy', '이탈리아'), (r'스페인|spain', '스페인'), (r'포르투갈|portugal', '포르투갈'),
    (r'아일랜드|ireland', '아일랜드'), (r'스웨덴|sweden', '스웨덴'), (r'노르웨이|norway', '노르웨이'),
    (r'덴마크|denmark', '덴마크'), (r'핀란드|finland', '핀란드'), (r'폴란드|poland', '폴란드'),
    (r'체코|czech', '체코'), (r'슬로바키아|slovak', '슬로바키아'), (r'헝가리|hungary', '헝가리'),
    (r'루마니아|romania', '루마니아'), (r'튀르키예|터키|turkey|t[üu]rkiye', '튀르키예'), (r'러시아|russia|moscow|모스크바', '러시아'),
    (r'우크라이나|ukraine', '우크라이나'),
    (r'아랍에미리트|\bu\.?a\.?e\b|united arab emirates|dubai|두바이|abu dhabi|아부다비', '아랍에미리트'), (r'사우디|saudi|soudi|riyadh|jeddah', '사우디아라비아'),
    (r'카타르|qatar', '카타르'), (r'쿠웨이트|kuwait', '쿠웨이트'), (r'이스라엘|israel', '이스라엘'), (r'이집트|egypt', '이집트'),
    (r'남아프리카|south africa', '남아프리카공화국'), (r'나이지리아|nigeria', '나이지리아'),
    (r'케이[맨먼만]|cayman', '케이맨제도'), (r'버진아일랜드|virgin islands|\bbvi\b', '영국령 버진아일랜드'),
    (r'버뮤다|bermuda', '버뮤다'), (r'마[샬셜]|marshall islands', '마셜제도'), (r'라이베리아|liberia', '라이베리아'),
    (r'라오스|laos|vientiane', '라오스'), (r'오만|\boman\b', '오만'), (r'요르단|jordan', '요르단'), (r'바레인|bahrain', '바레인'),
    (r'이라크|iraq', '이라크'), (r'이란|\biran\b', '이란'), (r'모잠비크|mozambique', '모잠비크'), (r'알제리|algeria', '알제리'),
    (r'가나|ghana', '가나'), (r'케냐|kenya', '케냐'), (r'모로코|morocco', '모로코'), (r'콩고|congo', '콩고'), (r'수단|sudan', '수단'),
    (r'불가리아|bulgaria', '불가리아'), (r'그리스|greece', '그리스'), (r'세르비아|serbia', '세르비아'), (r'크로아티아|croatia', '크로아티아'),
    (r'슬로베니아|slovenia', '슬로베니아'), (r'에스토니아|estonia', '에스토니아'), (r'리투아니아|lithuania', '리투아니아'),
    (r'라트비아|latvia', '라트비아'), (r'몰타|malta', '몰타'), (r'키프로스|cyprus', '키프로스'),
    (r'네팔|nepal', '네팔'), (r'동티모르|timor', '동티모르'), (r'피지|fiji', '피지'), (r'파푸아뉴기니|papua', '파푸아뉴기니'),
    (r'키리바시|kiribati', '키리바시'), (r'투발루|tuvalu', '투발루'), (r'나우루|nauru', '나우루'),
    (r'자메이카|jamaica', '자메이카'), (r'과테말라|guatemala', '과테말라'), (r'코스타리카|costa rica', '코스타리카'),
    (r'에콰도르|ecuador', '에콰도르'), (r'우루과이|uruguay', '우루과이'), (r'파라과이|paraguay', '파라과이'),
    (r'도미니카|dominican', '도미니카공화국'), (r'온두라스|honduras', '온두라스'), (r'엘살바도르|el salvador', '엘살바도르'),
    (r'니카라과|nicaragua', '니카라과'), (r'바하마|bahamas', '바하마'), (r'바베이도스|barbados', '바베이도스'),
    (r'저지|jersey(?!\s*city)', '저지'), (r'건지|guernsey', '건지'), (r'모리셔스|mauritius', '모리셔스'), (r'세이셸|seychelles', '세이셸'),
    (r'튀니지|tunisia', '튀니지'), (r'에티오피아|ethiopia', '에티오피아'), (r'탄자니아|tanzania', '탄자니아'),
    (r'아제르바이잔|azerbaijan', '아제르바이잔'), (r'조지아공화국|tbilisi', '조지아'), (r'키르기스|kyrgyz', '키르기스스탄'),
    (r'브루나이|brunei', '브루나이'), (r'마카오|macau|macao', '마카오'),
]
COUNTRY_RES = [(re.compile(p, re.I), c) for p, c in COUNTRIES]
US_STATE_RE = re.compile(
    r'(?i:\b(Alabama|Alaska|Arizona|Arkansas|California|Colorado|Connecticut|Delaware|Florida|Georgia|Hawaii|Idaho|Illinois|'
    r'Indiana|Iowa|Kansas|Kentucky|Louisiana|Maine|Maryland|Massachusetts|Michigan|Minnesota|Mississippi|Missouri|Montana|'
    r'Nebraska|Nevada|New Hampshire|New Jersey|New Mexico|New York|North Carolina|North Dakota|Ohio|Oklahoma|Oregon|'
    r'Pennsylvania|Rhode Island|South Carolina|South Dakota|Tennessee|Texas|Utah|Vermont|Virginia|Washington|'
    r'West Virginia|Wisconsin|Wyoming)\b)|\b(AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|ID|IL|IN|IA|KS|KY|LA|ME|MD|MA|MI|MN|MS|MO|'
    r'MT|NE|NV|NH|NJ|NM|NY|NC|ND|OH|OK|OR|PA|RI|SC|SD|TN|TX|UT|VT|VA|WA|WV|WI|WY)\s*\d{5}\b')


def country_of(text):
    """주소·소재지 문자열에서 나라를 찾는다. 주소는 끝에 나라가 오는 경우가 많아 뒤에서 가장 가까운 것을 고른다."""
    s = clean(text)
    if not s or s in ('-', '〃'):
        return ''
    if re.fullmatch(r'[가-힣 ]+', s):        # '베 트 남'처럼 한 글자씩 띄어 쓴 소재지
        s = s.replace(' ', '')
    hits = []
    for pattern, country in COUNTRY_RES:
        for m in pattern.finditer(s):
            hits.append((m.end(), country))
    if US_STATE_RE.search(s):
        hits.append((US_STATE_RE.search(s).end(), '미국'))
    if not hits:
        return ''
    # 한국 주소는 앞에 시·도가 오므로, 한국어로 시작하는 주소는 첫 번째 것을 쓴다
    if re.match(r'[가-힣]', s):
        return min(hits)[1]
    return max(hits)[1]

## 3. 절 나누기

사업의 내용은 공시 서식이 정해져 있어 절 제목(`<TITLE>`)으로 나눈다. 서식은 세 가지다.

| 서식 | 기업 | 절 |
|---|---|---|
| 제조 | 대부분 | 1. 사업의 개요, 2. 주요 제품 및 서비스, 3. 원재료 및 생산설비, 4. 매출 및 수주상황, 5. 위험관리 및 파생거래, 6. 주요계약 및 연구개발활동, 7. 기타 참고사항 |
| 금융 | 은행·보험·증권 | 1. 사업의 개요, 2. 영업의 현황, 3. 파생상품거래 현황, 4. 영업설비, 5. 재무건전성 등 |
| 혼합 | 금융 자회사가 있는 기업(현대차, 카카오, KT 등) | 제조 서식 7개 절(제목 앞에 "(제조서비스업)")과 금융 서식 5개 절("(금융업)") |

- 혼합 서식은 제조 서식 부분만 쓴다. 금융 자회사 쪽은 금리·규제 같은 업종 단위로 뉴스와 이어지기 때문이다.
- 금융 서식은 "1. 사업의 개요"만 쓴다. "2. 영업의 현황"은 회사당 중앙값 1.5만 자의 숫자 표(예수금, 대출 등)가 대부분이라 검색어 재료가 적다.

In [3]:
# ── 절 나누기: II. 사업의 내용을 절 제목으로 나누고 서식(제조·금융·혼합)을 정한다 ──
PART = r'^{n}\.\s*(\((?:제조서비스업|금융업)\))?\s*'
SECTION_PATTERNS = {       # 키: (정규식, 서식 쪽)
    's1': (PART.format(n=1) + r'사업의\s*개요', '제조서비스업'),
    's2': (PART.format(n=2) + r'주요\s*제품', '제조서비스업'),
    's3': (PART.format(n=3) + r'원재료', '제조서비스업'),
    's4': (PART.format(n=4) + r'매출', '제조서비스업'),
    's6': (PART.format(n=6) + r'주요\s*계약', '제조서비스업'),
    's7': (PART.format(n=7) + r'기타\s*참고', '제조서비스업'),
    'f1': (PART.format(n=1) + r'사업의\s*개요', '금융업'),
}
SECTION_NAMES = {'s1': '1. 사업의 개요', 's2': '2. 주요 제품 및 서비스', 's3': '3. 원재료 및 생산설비',
                 's4': '4. 매출 및 수주상황', 's6': '6. 주요계약 및 연구개발활동', 's7': '7. 기타 참고사항',
                 'f1': '1. 사업의 개요 (금융업)'}


def business_sections(xml):
    """II. 사업의 내용의 절을 {키: XML}로, 서식('제조'·'금융'·'혼합')과 함께 돌려준다."""
    titles = [(m.start(), clean(re.sub(r'<[^>]+>', '', m.group(1)))) for m in TITLE_RE.finditer(xml)]
    try:
        i = next(k for k, (_, t) in enumerate(titles) if re.match(r'^(II|Ⅱ)\.\s*사업의\s*내용', t))
        j = next(k for k, (_, t) in enumerate(titles) if k > i and re.match(r'^(III|Ⅲ)\.', t))
    except StopIteration:
        return {}, '없음'
    sub = titles[i + 1:j]
    has_fin = any(re.search(r'영업의\s*현황', t) for _, t in sub)
    has_mfg = any(re.search(r'주요\s*제품', t) for _, t in sub)
    fmt = '혼합' if has_fin and has_mfg else '금융' if has_fin else '제조'
    out = {}
    for k, (pos, t) in enumerate(sub):
        end = sub[k + 1][0] if k + 1 < len(sub) else titles[j][0]
        m = re.search(r'\((제조서비스업|금융업)\)', t)
        side = m.group(1) if m else ('금융업' if fmt == '금융' else '제조서비스업')
        for key, (pattern, want) in SECTION_PATTERNS.items():
            if side == want and re.match(pattern, t):
                out.setdefault(key, xml[pos:end])
    return out, fmt


docs = []
for r in selected.itertuples():
    xml = read_main(RAW / r.file)
    secs, fmt = business_sections(xml)
    docs.append(dict(stock_code=r.stock_code, rcept_no=r.latest_rcept_no, report=r.latest_report_nm,
                     fmt=fmt, sections=secs))
fmt_count = pd.Series([d['fmt'] for d in docs]).value_counts()
print('서식별 기업 수:', fmt_count.to_dict())
pd.DataFrame([{k: (k in d['sections']) for k in SECTION_PATTERNS} | {'fmt': d['fmt']} for d in docs]).groupby('fmt').sum()

서식별 기업 수: {'제조': 994, '금융': 52, '혼합': 17}


,s1,s2,s3,s4,s6,s7,f1
fmt,,,,,,,
금융,0,0,0,0,0,0,52
제조,994,994,994,994,994,994,0
혼합,17,17,17,17,17,17,17


## 4. 글로 바꾸고 줄이기

절 XML을 문서 순서대로 **단락**과 **표**로 바꾼다. 표는 `| 칸 | 칸 |` 모양의 줄로 쓰고, 옆으로 같은 값이 반복되는 병합 칸은 한 번만 쓴다. 그다음 검색어와 관계없는 부분을 뺀다.

**소제목 영역 빼기**: 소제목("가.", "(1)", "1)", "①"로 시작하는 단락의 앞부분)에 아래 말이 있으면 다음 소제목까지 뺀다. 소제목과 본문이 한 단락으로 붙어 있는 경우가 많아 단락 앞 40자로 판단한다.

| 절 | 빼는 영역 | 이유 |
|---|---|---|
| 3 | 생산능력, 생산실적, 가동률, 생산설비 현황, 설비 신설·매입, 투자 계획, 산출 근거 | 수량·장부가액이라 관계가 없다. 단 이 영역 안의 **사업장 표**(소재지·주소가 있는 표)는 남긴다 |
| 4 | 판매방법·조건, 판매조직, 조직 구성, 대금 회수 | 결제 조건(L/C, 어음)과 조직 설명이다. 판매경로, 주요 매출처, 수주상황, 판매전략은 남긴다 |
| 6 | 연구개발 비용·조직·인력, 연구개발 실적·활동 | 연구 과제 목록이 길다. **건강관리 업종은 연구개발 실적을 남긴다**(파이프라인이 곧 제품이고, 경쟁 약물이 적혀 있다) |

**그 밖에**

- 첫 열을 빼면 숫자뿐인 표(설비 장부가액, 연구개발비 등)는 뺀다.
- 긴 표는 앞 20줄(2절은 60줄)만 남기고 "아래 N행 생략"이라고 적는다.
- 1·7절은 통째로 넣지 않는다. "시장점유율", "점유율", "경쟁사", "경쟁업체", "경쟁요소", "경쟁상황"이 나오는 단락과 그 뒤 블록(다음 소제목 전까지, 최대 6개)만 잘라 온다. 절마다 2,500자까지다.

In [4]:
# ── 글로 바꾸고 줄이기: 단락·표를 글로 바꾸고, 빼는 영역·숫자만 있는 표·긴 표를 줄인다 ──
def blocks(section_xml):
    """절 XML → 문서 순서의 블록 목록 [('title'|'p'|'table', 글 또는 칸 목록)]"""
    soup = BeautifulSoup(section_xml, 'lxml')
    out = []
    for el in soup.find_all(['title', 'p', 'table']):
        if el.name == 'p' and el.find_parent('table'):
            continue
        if el.name == 'table':
            if el.find('table'):           # 표를 감싼 바깥 표는 건너뛰고 안쪽 표만 읽는다
                continue
            grid = table_grid(el)
            if grid:
                out.append(('table', grid))
            continue
        text = clean(el.get_text(' '))
        if text:
            out.append(('title' if el.name == 'title' else 'p', text))
    return out


def numeric_only(grid):
    """첫 열(항목 이름)을 빼면 모두 숫자인 표"""
    cells = [c for r in grid[1:] for c in r[1:] if c]
    return len(cells) >= 6 and all(NUM_CELL.match(c) for c in cells)


def render(bl, drop_numeric=True):
    """블록 목록 → 글. (글, 뺀 숫자 표 수)"""
    lines, dropped = [], 0
    for kind, v in bl:
        if kind == 'table':
            if drop_numeric and numeric_only(v):
                dropped += 1
                continue
            rows = ['| ' + ' | '.join(c for k, c in enumerate(r) if k == 0 or c != r[k - 1]) + ' |' for r in v]
            lines.append('\n'.join(rows))
        elif kind == 'title':
            continue            # 절 제목은 입력 파일에서 '### [절 이름]'으로 따로 단다
        else:
            lines.append(v)
    return '\n'.join(lines), dropped


HEADING_RE = re.compile(r'^([가-하]\.|\(\d+\)|\d+\)|[①-⑳]|\d+\.)\s*\S')
DROP_ZONES = {
    's3': r'생산\s*능력|생산\s*실적|가동률|생산\s*설비의?\s*현황|설비의?\s*(신설|매입)|투자\s*계획|시설\s*투자|신규\s*투자|감가상각|산출\s*근거',
    's4': r'판매\s*방법|판매\s*조건|판매\s*조직|조직\s*구성|대금\s*회수',
    's6': r'연구\s*개발\s*비용|연구\s*개발비|담당\s*조직|연구\s*조직|연구\s*인력|연구소\s*현황|연구\s*개발\s*(실적|활동|과제)|연구\s*실적',
}
DROP_ZONES_BIO_S6 = r'연구\s*개발\s*비용|연구\s*개발비|담당\s*조직|연구\s*조직|연구\s*인력|연구소\s*현황'
KEEP_SITE_TABLE = re.compile(r'소재지|주소|사업장')


def heading_head(kind, v):
    """소제목이면 앞부분(최대 40자)을, 아니면 None"""
    if kind == 'title':
        return v
    if kind == 'p' and HEADING_RE.match(v):
        return v[:40]
    return None


def trim(bl, key, bio=False):
    """빼는 영역을 지우고 긴 표는 앞부분만 남긴다. (블록 목록, 뺀 블록 수)"""
    pattern = DROP_ZONES_BIO_S6 if (key == 's6' and bio) else DROP_ZONES.get(key)
    max_rows = MAX_ROWS_PRODUCTS if key == 's2' else MAX_ROWS
    out, dropping, n = [], False, 0
    for kind, v in bl:
        head = heading_head(kind, v)
        if head is not None:
            dropping = bool(pattern and re.search(pattern, head))
        if dropping:
            if kind == 'table' and key == 's3' and KEEP_SITE_TABLE.search(' '.join(v[0])):
                out.append((kind, v))
            else:
                n += 1
            continue
        if kind == 'table' and len(v) > max_rows + 1:
            v = v[:max_rows + 1] + [[f'…(아래 {len(v) - max_rows - 1}행 생략)']]
        out.append((kind, v))
    return out, n


SHARE_RE = re.compile(r'시장\s*점유율|점유율|경쟁\s*(사|업체|요소|상황|현황|제품|구도)|주요\s*경쟁')


def share_windows(bl):
    """시장점유율·경쟁이 나오는 단락과 그 뒤 블록(다음 소제목 전까지, 최대 6개). 절마다 SHARE_WINDOW_CHARS자까지"""
    picked, i = [], 0
    while i < len(bl):
        kind, v = bl[i]
        if kind != 'table' and SHARE_RE.search(v):
            win, j = [bl[i]], i + 1
            while j < len(bl) and len(win) < 6:
                k2, v2 = bl[j]
                if k2 == 'title' or (k2 == 'p' and heading_head(k2, v2) and not SHARE_RE.search(v2[:40])):
                    break
                win.append(bl[j])
                j += 1
            picked.append(win)
            i = j
        else:
            i += 1
    out, size = [], 0
    for win in picked:
        text, _ = render(win, drop_numeric=False)
        if size + len(text) > SHARE_WINDOW_CHARS:
            text = text[:max(0, SHARE_WINDOW_CHARS - size)] + ' …(이하 생략)'
        out.append(text)
        size += len(text)
        if size >= SHARE_WINDOW_CHARS:
            break
    return '\n\n'.join(out)

## 5. LLM 입력 파일

회사마다 `inputs/종목코드.txt` 하나를 쓴다. 맨 위에 회사 정보를 적고, 부분마다 `### [절 이름]` 제목을 단다.

- 내용이 바뀐 파일만 다시 쓴다. 이 폴더는 iCloud로 동기화되는 바탕화면 아래에 있어, 파일을 지웠다가 같은 이름으로 다시 쓰면 동기화가 충돌로 보고 `000020 2.txt` 같은 사본을 만든다.
- 실행 끝에 대상에서 빠진 회사의 파일과 이런 사본을 지운다.

| 서식 | 넣는 부분 |
|---|---|
| 제조·혼합 | 2절, 3절, 4절, 6절(4장의 규칙으로 줄임) + 1·7절의 시장점유율·경쟁 단락 |
| 금융 | 1. 사업의 개요 |

`inputs_index.csv`에 부분별 글자 수와 뺀 영역 수를 남긴다. LLM 실행 전에 구독 사용량을 추정하는 데 쓴다.

In [5]:
# ── LLM 입력 파일: 회사마다 inputs/종목코드.txt 하나(맨 위에 회사 정보·접수번호·기준일) ──
index_rows, written = [], set()
for d in docs:
    code_, secs = d['stock_code'], d['sections']
    co = companies.loc[code_] if code_ in companies.index else None
    name = co['stock_name'] if co is not None else ''
    sector = co['wi26_sector'] if co is not None else ''
    bio = sector == '건강관리'
    parts, row = [], dict(stock_code=code_, stock_name=name, wi26_sector=sector, fmt=d['fmt'],
                          rcept_no=d['rcept_no'], report=d['report'])
    if d['fmt'] == '금융':
        if 'f1' in secs:
            text, _ = render(blocks(secs['f1']))
            parts.append(('f1', text))
    else:
        cut_blocks = cut_tables = 0
        for key in ['s2', 's3', 's4', 's6']:
            if key not in secs:
                continue
            bl, n = trim(blocks(secs[key]), key, bio=bio)
            text, dropped = render(bl)
            parts.append((key, text))
            cut_blocks += n
            cut_tables += dropped
        for key in ['s1', 's7']:
            if key in secs:
                text = share_windows(blocks(secs[key]))
                if text:
                    parts.append((key, text))
        row.update(cut_blocks=cut_blocks, cut_numeric_tables=cut_tables)
    header = (f"# {name} ({code_}) {d['report']}\n"
              f"접수번호: {d['rcept_no']} · 기준일: {period_end(d['report'])} · 업종(WI26): {sector} · 서식: {d['fmt']}\n")
    body = '\n\n'.join(f"### [{SECTION_NAMES[k]}{' · 시장점유율·경쟁 부분' if k in ('s1', 's7') else ''}]\n{t}"
                       for k, t in parts if t)
    path, content = INPUTS / f'{code_}.txt', header + '\n' + body + '\n'
    if not path.exists() or path.read_text(encoding='utf-8') != content:   # 바뀐 파일만 쓴다
        path.write_text(content, encoding='utf-8')
    written.add(path.name)
    for k in SECTION_NAMES:
        row[f'chars_{k}'] = sum(len(t) for kk, t in parts if kk == k)
    row['chars_total'] = len(header) + len(body)
    index_rows.append(row)

# 대상에서 빠진 회사의 파일, iCloud 동기화가 만든 사본('000020 2.txt')을 지운다
stale = [p for p in INPUTS.glob('*.txt') if p.name not in written]
for p in stale:
    p.unlink()
inputs_index = pd.DataFrame(index_rows)
inputs_index.to_csv(OUT / 'inputs_index.csv', index=False, encoding='utf-8-sig')
print('입력 파일', len(inputs_index), '개 (지운 파일', len(stale), '개) / 전체', f"{inputs_index['chars_total'].sum():,}", '자 / 회사당 중앙값',
      f"{int(inputs_index['chars_total'].median()):,}", '자')

입력 파일 1063 개 (지운 파일 0 개) / 전체 11,422,158 자 / 회사당 중앙값 8,121 자


## 6. 표 규칙 추출

열 제목(머리글)으로 열을 고른다. 머리글은 숫자 칸이 없는 앞쪽 줄(최대 3줄)이고, "(단위 : 백만원)" 같은 줄은 단위로 따로 남긴다. 합계·소계·총계·연결조정·내부거래 줄은 남기지 않는다.

| 표 | 찾는 조건 | 열 |
|---|---|---|
| `segments.csv` (2절) | 제품 열(품목, 제품, 품명, 서비스)과 매출 열(매출액, 금액) 또는 비중 열이 있는 **첫 번째** 표 | 부문, 제품, 용도, 상표, 매출액, 비중. "97,146,675(100%)"처럼 매출액과 비율을 한 칸에 적은 표도 읽는다 |
| `materials.csv` (3절) | 품목 열(품목, 원재료, 품명)과 매입처 열 또는 매입액 열이 있는 표 | 부문, 품목, 용도, 매입액, 비중, 매입처(원문, 나눈 목록) |
| `sites.csv` (3절) | 소재지·주소 열이 있는 표 | 구분, 사업장, 주소, 나라 |

- 비중이 0~100을 벗어나면 칸을 잘못 읽은 것이라 빈칸으로 둔다.
- 매입처 칸은 쉼표·"및"·"/"로 나누고 "등", 각주 표시를 지운다(예: `삼성전자(*), SONY` → `삼성전자`, `SONY`). 이름 통일(#16)은 하지 않는다.
- 금융 서식 기업은 이 표들이 없어 건너뛴다.

In [6]:
# ── 표 규칙 추출: 열 제목으로 열을 골라 사업부문 매출, 원재료·매입처, 사업장을 읽는다 ──
UNIT_RE = re.compile(r'단위\s*[:：]?\s*([^)\]]*)')
TOTAL_RE = re.compile(r'^(합\s*계|총\s*계|계|소\s*계|총\s*합\s*계|연결\s*조정|내부\s*거래|조\s*정)$|(합\s*계|총\s*계|소\s*계)$')


def is_total(*texts):
    """합계·소계 줄인가."""
    return any(TOTAL_RE.search(t.replace(' ', '')) for t in texts if t)


def split_head(grid):
    """머리글(숫자 칸이 없는 앞쪽 줄, 최대 3줄)과 본문으로 나눈다. (열 제목, 본문, 단위)"""
    unit, rows = '', list(grid)
    while rows and len(set(c for c in rows[0] if c)) <= 2 and any(UNIT_RE.search(c) or re.match(r'^\[.*\]$', c) for c in rows[0]):
        m = next((UNIT_RE.search(c) for c in rows[0] if UNIT_RE.search(c)), None)
        unit = unit or (clean(m.group(1)) if m else '')
        rows = rows[1:]
    if not rows:
        return [], [], unit
    n = 0
    for r in rows:
        if any(NUM_CELL.match(c) and not re.fullmatch(r'\d{4}', c) for c in r[1:] if c):
            break
        n += 1
    # 숫자 칸이 하나도 없는 표(사업장 표 등)는 첫 줄만 머리글로 본다
    n = 1 if n == len(rows) else max(1, min(n, 3))
    width = max(len(r) for r in rows)
    labels = [''.join(dict.fromkeys(r[i] for r in rows[:n] if i < len(r) and r[i])).replace(' ', '') for i in range(width)]
    return labels, rows[n:], unit


def find_col(labels, pattern, exclude=None, used=()):
    """열 제목이 pattern에 맞는 첫 열 번호(exclude에 맞거나 이미 쓴 열은 건너뛴다). 없으면 None."""
    for i, lab in enumerate(labels):
        if i not in used and re.search(pattern, lab) and not (exclude and re.search(exclude, lab)):
            return i
    return None


def cell(r, i):
    """줄 r의 i번째 칸. 없으면 ''."""
    return r[i] if i is not None and i < len(r) else ''


def pct(v):
    return v if v is not None and 0 <= v <= 100 else None


COMBINED_RE = re.compile(r'^([\d,\.\-△()]+)\s*\(\s*([\d\.]+)\s*%?\s*\)$')     # '97,146,675(100%)'


def segment_rows(bl):
    """사업부문·제품별 매출액·매출 비중 표의 줄을 읽는다."""
    for kind, grid in bl:
        if kind != 'table' or len(grid) < 2:
            continue
        labels, body, unit = split_head(grid)
        combined = find_col(labels, r'매출.*(비율|비중)')
        rev = combined if combined is not None else find_col(labels, r'매출|금액', exclude=r'비중|비율|유형')
        share = None if combined is not None else find_col(labels, r'비중|비율|구성비')
        prod = find_col(labels, r'품목|제품|품명|서비스|상품')
        if prod is None or (rev is None and share is None):
            continue
        seg = find_col(labels, r'사업부문|부문|사업', used={prod})
        use = find_col(labels, r'용도')
        brand = find_col(labels, r'상표|브랜드')
        out = []
        for r in body:
            if is_total(cell(r, seg), cell(r, prod), r[0]):
                continue
            rv, sh = cell(r, rev), cell(r, share)
            m = COMBINED_RE.match(clean(rv))
            if m:
                rv, sh = m.group(1), m.group(2)
            if not clean(cell(r, prod)) or (to_num(rv) is None and to_num(sh) is None):
                continue
            out.append(dict(segment=cell(r, seg), product=cell(r, prod), use=cell(r, use), brand=cell(r, brand),
                            revenue=to_num(rv), share_pct=pct(to_num(sh)), unit=unit))
        if out:
            return out          # 첫 번째 매출 표만 쓴다(뒤의 표는 세부·전기 표인 경우가 많다)
    return []


def split_names(text):
    """'SHOEI, GUANGBO 등' → ['SHOEI', 'GUANGBO']"""
    s = re.sub(r'\(\s*\*\d*\s*\)|\*\d*|\s등$|\s등\s', ' ', clean(text))
    parts = re.split(r'\s*[,，、/;]\s*|\s+및\s+', s)
    return [p.strip(' .') for p in parts if p.strip(' .') and p.strip() not in ('-', '－', '등', '기타')]


def material_rows(bl):
    """원재료 표의 줄을 읽는다(품목, 매입액, 매입 비중, 매입처)."""
    out = []
    for kind, grid in bl:
        if kind != 'table' or len(grid) < 2:
            continue
        labels, body, unit = split_head(grid)
        sup = find_col(labels, r'매입처|구입처|공급처|업체|거래처|공급사|조달처')
        item = find_col(labels, r'품목|원재료|품명|원자재|자재명', used={sup} - {None})
        amt = find_col(labels, r'매입액|금액|투입액|구입액')
        share = find_col(labels, r'비중|비율|구성비')
        if item is None or (sup is None and amt is None):
            continue
        seg = find_col(labels, r'사업부문|부문', used={item})
        use = find_col(labels, r'용도')
        for r in body:
            it = cell(r, item)
            if not clean(it) or is_total(it, cell(r, seg)):
                continue
            out.append(dict(segment=cell(r, seg), item=it, use=cell(r, use), amount=to_num(cell(r, amt)),
                            share_pct=pct(to_num(cell(r, share))), suppliers_raw=cell(r, sup),
                            suppliers=' ; '.join(split_names(cell(r, sup))) if sup is not None else '', unit=unit))
    return out


def site_rows(bl):
    """사업장(공장·연구소 등) 표의 줄을 읽는다(구분, 이름, 주소)."""
    out = []
    for kind, grid in bl:
        if kind != 'table' or len(grid) < 2:
            continue
        labels, body, _ = split_head(grid)
        addr = find_col(labels, r'소재지|주소|위치')
        if addr is None:
            continue
        name = find_col(labels, r'사업장|공장|사업소|법인|명칭', used={addr})
        kind_col = find_col(labels, r'구분|지역|용도|형태', used={addr, name} - {None})
        for r in body:
            a, site, k = cell(r, addr), cell(r, name), cell(r, kind_col)
            if not clean(a) or clean(a) in ('-', '－') or is_total(a, site, k) or ASSET_RE.search(k + ' ' + site):
                continue
            out.append(dict(kind=k, site=site, address=a, country=site_country(a, site, k)))
    return out


# 유형자산 표(소재지 열이 있는)의 자산 종류 줄은 사업장이 아니다
ASSET_RE = re.compile(r'차량\s*운반구|공구|기구|비품|기계\s*장치|구축물|건설\s*중|토\s*지$|^건\s*물$|사용권|상세\s*내역')
# 한글 주소가 시·군·구·읍·면·동·리·로·길로 끝나는 말을 포함하면 국내로 본다(예: '청주', '함안군북장백로228')
KR_PLACE_RE = re.compile(r'[가-힣]{1,6}(시|군|구|읍|면|동|리|로|길)(\s|\d|$|,|\(|外|외|등)|^전\s*국$')
KR_CITIES = ('청주', '천안', '아산', '평택', '이천', '오송', '당진', '구미', '포항', '울산', '창원', '여수', '광양', '군산',
             '익산', '안산', '시흥', '화성', '용인', '김천', '거제', '진천', '음성', '오창', '논산', '원주', '평창', '대산')


def site_country(address, site, kind):
    """주소 → 사업장 이름 → 구분 순서로 나라를 찾는다. 못 찾으면 국내 지명 규칙으로 본다."""
    found = country_of(address) or country_of(site) or country_of(kind)
    if found:
        return found
    text = f'{address} {site}'
    if not re.search(r'[A-Za-z]{4,}', address) and (KR_PLACE_RE.search(text) or any(c in text for c in KR_CITIES)):
        return '대한민국'
    return ''


seg_all, mat_all, site_all = [], [], []
for d in docs:
    if d['fmt'] == '금융':
        continue
    base = dict(stock_code=d['stock_code'], rcept_no=d['rcept_no'], base_date=period_end(d['report']))
    secs = d['sections']
    if 's2' in secs:
        seg_all += [base | x for x in segment_rows(blocks(secs['s2']))]
    if 's3' in secs:
        bl3 = blocks(secs['s3'])
        mat_all += [base | x for x in material_rows(bl3)]
        site_all += [base | x for x in site_rows(bl3)]

names = companies['stock_name']
segments = pd.DataFrame(seg_all)
materials = pd.DataFrame(mat_all)
sites = pd.DataFrame(site_all)
for t in (segments, materials, sites):
    t.insert(1, 'stock_name', t['stock_code'].map(names))
segments.to_csv(TABLES / 'segments.csv', index=False, encoding='utf-8-sig')
materials.to_csv(TABLES / 'materials.csv', index=False, encoding='utf-8-sig')
sites.to_csv(TABLES / 'sites.csv', index=False, encoding='utf-8-sig')
print('segments', len(segments), '줄 / materials', len(materials), '줄 / sites', len(sites), '줄')

segments 4402 줄 / materials 5796 줄 / sites 10132 줄


## 7. 결과 확인

**입력 크기**: 다음 노트북에서 LLM 사용량을 추정하는 기준이다. 글자 수는 표의 `|` 구분 기호를 포함한다.

In [7]:
# ── 확인: 입력 크기(다음 노트북의 LLM 사용량 어림 기준) ──
size_cols = [c for c in inputs_index.columns if c.startswith('chars_s') or c.startswith('chars_f')]
summary = pd.DataFrame({
    '전체(만 자)': (inputs_index[size_cols].sum() / 1e4).round(1),
    '들어간 기업 수': (inputs_index[size_cols] > 0).sum(),
}).rename(index=lambda c: SECTION_NAMES[c.replace('chars_', '')])
print('전체', f"{inputs_index['chars_total'].sum() / 1e4:,.0f}만 자 / 회사당 중앙값",
      f"{int(inputs_index['chars_total'].median()):,}자, 상위 10%",
      f"{int(inputs_index['chars_total'].quantile(0.9)):,}자 이상, 최대 {int(inputs_index['chars_total'].max()):,}자")
summary

전체 1,142만 자 / 회사당 중앙값 8,121자, 상위 10% 19,997자 이상, 최대 79,882자


,전체(만 자),들어간 기업 수
1. 사업의 개요,33.3,262
2. 주요 제품 및 서비스,189.1,1010
3. 원재료 및 생산설비,268.2,1011
4. 매출 및 수주상황,248.7,1011
6. 주요계약 및 연구개발활동,222.1,811
7. 기타 참고사항,142.3,797
1. 사업의 개요 (금융업),15.8,52


In [8]:
# ── 확인: 입력이 긴 회사 ──
# 입력이 긴 기업 (지주사, 바이오가 많다)
inputs_index.sort_values('chars_total', ascending=False).head(10)[
    ['stock_code', 'stock_name', 'wi26_sector', 'fmt', 'chars_total', 'chars_s2', 'chars_s3', 'chars_s4', 'chars_s6']]

,stock_code,stock_name,wi26_sector,fmt,chars_total,chars_s2,chars_s3,chars_s4,chars_s6
966,115180,큐리언트,건강관리,제조,79882,49735,1121,1324,27516
463,001630,종근당홀딩스,건강관리,제조,78990,5894,5723,5509,59126
139,004990,롯데지주,"상사,자본재",제조,70504,12546,25078,19364,9993
52,267250,HD현대,조선,제조,68117,9969,21230,14973,19211
71,078930,GS,에너지,제조,64165,23098,16292,13927,7350
213,004800,효성,"상사,자본재",제조,64122,7414,30955,13869,9150
180,003090,대웅,건강관리,제조,61701,3888,9804,5435,39297
839,347850,디앤디파마텍,건강관리,제조,58629,14531,2507,2921,37053
43,003550,LG,"상사,자본재",제조,57413,14194,20575,11350,8566
697,007460,에이프로젠,건강관리,제조,55979,6451,3825,4823,38149


**표가 규칙으로 읽힌 정도**: 제조·혼합 서식 기업 중 각 표가 한 줄 이상 나온 기업의 비율이다. 읽히지 않은 기업도 표는 LLM 입력에 들어 있다. 매출 비중 합이 90~110%인 기업은 매출 표를 온전히 읽은 것으로 본다.

In [9]:
# ── 확인: 표가 규칙으로 읽힌 정도 ──
targets = inputs_index[inputs_index['fmt'] != '금융']
seg_share = segments.groupby('stock_code')['share_pct'].sum()
coverage = pd.DataFrame({
    '기업 수': [
        len(targets),
        segments['stock_code'].nunique(),
        int(seg_share.between(90, 110).sum()),
        materials['stock_code'].nunique(),
        materials.loc[materials['suppliers'] != '', 'stock_code'].nunique(),
        sites['stock_code'].nunique(),
    ]}, index=['제조·혼합 서식 기업', '매출 표 읽힘', '　매출 비중 합 90~110%', '원재료 표 읽힘', '　실명 매입처 있음', '사업장 표 읽힘'])
coverage['비율'] = (coverage['기업 수'] / len(targets) * 100).round(0).astype(int).astype(str) + '%'
coverage

,기업 수,비율
제조·혼합 서식 기업,1011,100%
매출 표 읽힘,769,76%
매출 비중 합 90~110%,499,49%
원재료 표 읽힘,652,64%
실명 매입처 있음,287,28%
사업장 표 읽힘,568,56%


In [10]:
# ── 확인: 회사 하나의 입력 글 ──
# 예: 삼성전기
show = '009150'
print((INPUTS / f'{show}.txt').read_text(encoding='utf-8')[:1500])

# 삼성전기 (009150) 사업보고서 (2025.12)
접수번호: 20260310003071 · 기준일: 2025-12-31 · 업종(WI26): IT하드웨어 · 서식: 제조

### [2. 주요 제품 및 서비스]
가. 주요 제품 등의 현황당사는 스마트폰, 전장용, PC, 디스플레이, TV 부품 등을 생산, 판매하고 있습니다.2025년 사업부문별 매출액 및 매출비중은 다음과 같습니다.
| [제53기 ('25.01.01 ~ '25.12.31)] | (단위 : 백만원) |
| 사업부문 | 주요제품 | 구체적용도 | 매출액 | 비중 |
| 컴포넌트 | 수동소자(MLCC, Inductor, Chip Resistor 등) | 스마트폰용, 전장용, PC용 등 | 5,198,501 | 45.95% |
| 패키지솔루션 | 반도체패키지기판 | PC용, 스마트폰용 등 | 2,301,753 | 20.34% |
| 광학솔루션 | 카메라모듈 | 스마트폰용, 전장용 등 | 3,814,205 | 33.71% |
| 계 | 11,314,459 | 100.00% |
※ 매출액은 연결기준으로 작성되었으며, 세부 유형별 매출액은 'Ⅱ. 사업의 내용 4. 매출 및 수주상황'을 참고하시기 바랍니다. 나. 주요 제품 등의 가격변동추이 MLCC의 평균판매가격은 전년 대비 0.9% 상승하였으며, 반도체패키지기판의 평균판매가격은 전년 대비 4.1% 상승하였습니다. 카메라모듈의 평균판매가격도 전년 대비 14.0% 상승하였습니다.

### [3. 원재료 및 생산설비]
당사는 국내 및 해외 원자재 업체로부터 제품생산에 필요한 원재료를 매입하고 있으며, 주요 원재료 등의 현황 및 가격변동추이는 다음과 같습니다.
가. 주요 원재료 등의 현황
| [제53기 ('25.01.01 ~ '25.12.31)] | (단위 : 백만원) |
| 사업부문 | 매입유형 | 품목 | 용도 | 매입액 | 비중 | 업체명(매입처) |
| 컴포넌트 | 원자재 | PASTE/POWDER | 콘덴서 주재료 | 664,492 | 49.3% |

In [11]:
# ── 확인: 회사 하나의 규칙 추출 표 ──
display(segments[segments['stock_code'] == show][['segment', 'product', 'use', 'revenue', 'share_pct', 'unit']])
display(materials[materials['stock_code'] == show][['segment', 'item', 'share_pct', 'suppliers']])
sites[sites['stock_code'] == show][['kind', 'site', 'country']]

,segment,product,use,revenue,share_pct,unit
14,컴포넌트,"수동소자(MLCC, Inductor, Chip Resistor 등)","스마트폰용, 전장용, PC용 등",5198501.0,45.95,
15,패키지솔루션,반도체패키지기판,"PC용, 스마트폰용 등",2301753.0,20.34,
16,광학솔루션,카메라모듈,"스마트폰용, 전장용 등",3814205.0,33.71,


,segment,item,share_pct,suppliers
26,컴포넌트,PASTE/POWDER,49.3,SHOEI ; GUANGBO
27,컴포넌트,FILM,25.1,TORAY ; COSMO
28,컴포넌트,기타,25.6,
29,패키지솔루션,CCL/PPG,21.4,MITSUBISHI ; RESONAC
30,패키지솔루션,약품류,16.4,ATOTECH ; NR G&C
31,패키지솔루션,기타,62.2,
32,광학솔루션,센서 IC,49.4,삼성전자 ; SONY
33,광학솔루션,ACTUATOR,21.1,ACTRO ; HAESUNG OPTICS
34,광학솔루션,기타,29.5,


,kind,site,country
56,국내,수원,대한민국
57,국내,세종,대한민국
58,국내,부산,대한민국
59,해외생산,천진,중국
60,해외생산,고신,중국
61,해외생산,필리핀,필리핀
62,해외생산,베트남,베트남
63,해외판매,미국,미국
64,해외판매,독일,독일
65,해외판매,싱가폴,싱가포르


In [12]:
# ── 확인: 사업장 나라 분포 ──
# 사업장 나라 분포 (대한민국 제외 상위)
sites.loc[sites['country'] != '대한민국', 'country'].replace('', '(못 정함)').value_counts().head(15)

country
(못 정함)    1030
중국         371
미국         331
베트남        199
인도          64
인도네시아       63
일본          59
독일          50
싱가포르        44
필리핀         38
태국          33
대만          29
호주          26
멕시코         26
카자흐스탄       23
Name: count, dtype: int64